You have at your disposal 100000 images of human faces, and their occlusion label.
The goal of this challenge is to regress the percentage of the face that is occluded.
We also want to have similar performances on female and male, the gender label is given for the train database

Below is the formula of the evaluation score

$$
 Err = \frac{\sum_{i}{w_i(p_i - GT_i)^2}}{\sum_{i}{w_i}}, w_i = \frac{1}{30} + GT_i
$$

$$
Score = \frac{Err_F + Err_M}{2} + \left | Err_F - Err_M \right |
$$

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
from collections import OrderedDict

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

import cv2
import os

### Load dataframes

In [ ]:
from pathlib import Path

if not Path("test1_predictions.csv").exists():
    raise FileNotFoundError(
        "results_part1.csv introuvable. Exécuter d'abord le notebook 1."
    )

In [ ]:
df_test = pd.read_csv("../../occlusion_datasets/test_students.csv", delimiter=',')

image_dir = "../../crops/Crop_224_5fp_100K"

In [ ]:
df_test.head()

#### Remove nan values

In [ ]:
df_test = df_test.dropna()
df_test["row_id"] = range(len(df_test))

### Split Dataframe in test1 and test2

In [ ]:
split_idx = len(df_test) // 2
df_test1 = df_test.iloc[:split_idx].reset_index()
df_test2 = df_test.iloc[split_idx:].reset_index()

In [ ]:
len(df_test1), len(df_test2), len(df_test)

### Check that all images are read correctly

In [ ]:
for idx, row in tqdm(df_test2.iterrows(), total=len(df_test1)):
    try:
        filename = df_test1.loc[idx, 'filename']
        img2display = Image.open(f"{image_dir}/{filename}")
    except ValueError as e:
        print(idx, e)

### Make Dataset and Dataloader

In [ ]:
class Dataset(torch.utils.data.Dataset):
    'Characterizes a dataset for PyTorch'
    def __init__(self, df, image_dir, training=True):
         'Initialization'
         self.training = training
         self.image_dir = image_dir
         self.df = df
         self.transform = transforms.ToTensor()
         
    def __len__(self):
        'Denotes the total number of samples'
        return len(self.df)

    def __getitem__(self, index):
        'Generates one sample of data'
        # Select sample
        row = self.df.loc[index]
        filename = row['filename']

        # Load data and get label
        img = Image.open(f"{image_dir}/{filename}")

        X = cv2.imread(f"{image_dir}/{filename}")

        if self.training:
            y = row['FaceOcclusion']
            y = np.float32(y)
            gender = row['gender']
            return X, y, gender, filename
        else:
            y = row['row_id']
            gender = None
            return X, y, filename

# Training-Free track

## Installation

In [ ]:
from pathlib import Path
import subprocess
import sys

import gdown

### 3DDFA-V2

In [ ]:
repo_dir = Path("3DDFA_V2")

if not repo_dir.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/cleardusk/3DDFA_V2.git"],
        check=True
    )

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", "3DDFA_V2/requirements.txt"],
        check=True
    )

    print("Repository cloné et dépendances installées.")
else:
    print("Repository déjà présent, installation ignorée.")

### BiSeNet

In [ ]:
repo_dir = Path("face-parsing.PyTorch")

if not repo_dir.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/zllrunning/face-parsing.PyTorch.git"],
        check=True
    )

    print("Repository cloné.")
else:
    print("Repository déjà présent, installation ignorée.")

Le modèle pré-entrainé n'est pas directement accessible depuis le repo Git, mais depuis un Drive. Voici les informations pour le récupérer en cas d'échec du téléchargement automatique.

Lien pour la copie des poids : https://drive.google.com/open?id=154JgKpzCPW82qINcVieuPH3fZ2e0P812
A placer dans un sous-dossier weights, à partir du dossier courant du notebook

In [ ]:
Path("weights").mkdir(exist_ok=True)

file_id = "154JgKpzCPW82qINcVieuPH3fZ2e0P812"
output = "weights/79999_iter.pth"

if not Path(output).exists():
    gdown.download(
        f"https://drive.google.com/uc?id={file_id}",
        output,
        quiet=False
    )
    print("Modèle chargé")
else:
    print("Modèle déjà installé")

## Model Imports

In [ ]:
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device ='cpu'
print(device)

In [ ]:
#Face Analysis

from insightface.app import FaceAnalysis

app = FaceAnalysis(
    name='buffalo_l',
    providers=[
        'CPUExecutionProvider'
    ]
)
app.prepare(
    ctx_id=0,
    det_size=(224, 224)
)

In [ ]:
# 3DDFA-V2

ROOT = Path.cwd()
THREEDDFA_PATH = ROOT / "3DDFA_V2"
sys.path.append(str(THREEDDFA_PATH))
print(THREEDDFA_PATH)

from TDDFA import TDDFA

MODEL_PATH = Path.cwd() / "3DDFA_V2" / "weights" / "mb1_120x120.pth"

cfg = {
    'arch': 'mobilenet',
    'checkpoint_fp': str(MODEL_PATH),
    'gpu_mode': False
}

tddfa = TDDFA(**cfg)

from utils.pose import viz_pose

In [ ]:
#BiSeNet

BISENET_PATH = ROOT / "face-parsing.PyTorch"
sys.path.append(str(BISENET_PATH))

from model import BiSeNet
import torch

WEIGHTS = (
    Path.cwd()
    / "weights"
    / "79999_iter.pth"
)

assert WEIGHTS.exists()

n_classes = 19
net = BiSeNet(n_classes=n_classes)
net.load_state_dict(
    torch.load(
        WEIGHTS,
        map_location="cpu"
    )
)

net.eval().to(device)

In [ ]:
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation

seg_processor = SegformerImageProcessor.from_pretrained("jonathandinu/face-parsing")
seg_model = SegformerForSemanticSegmentation.from_pretrained("jonathandinu/face-parsing")
seg_model.to(device)

## Pipeline and configuration

In [ ]:
from torchvision import transforms

# Segmentation classes for BiSeNet and SegFormer

VISIBLE_FACE_CLASSES_B = [
    0,   # background (in case of missed detection)
    1,   # skin
    2,   # left eyebrow
    3,   # right eyebrow
    4,   # left eye
    5,   # right eye
    6,   # eyeglasses
    10,  # nose
    11,  # mouth
    12,  # upper lip
    13  # lower lip
]

VISIBLE_FACE_CLASSES_S = [
    1,   # skin
    2,  # nose
    3,   # eyeglasses
    4,   # left eye
    5,   # right eye
    6,   # left eyebrow
    7,   # right eyebrow
    10,  # mouth
    11,  # upper lip
    12  # lower lip
]

HAIR_B = [17]
HAT_B = [18]
HAIR_S = [13]
HAT_S = [14]

# Weights to adjust the different cases (gender, presence of hair, hats and other objects)
F_WEIGHTS = {"hair_bi": 0.376, "hat_bi": 0.425, "other_bi": 0.478,
             "hair_sf": 0.619, "hat_sf": 0.902, "other_bg_sf": 1.087}
M_WEIGHTS = {"hair_bi": 0.489, "hat_bi": 0.294, "other_bi": 0.210,
             "hair_sf": 0.484, "hat_sf": 0.609, "other_bg_sf": 0.382}

# Transformation in tensor
to_tensor = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.485, 0.456, 0.406),
        (0.229, 0.224, 0.225)
    ),
])

In [ ]:
def overlay_mask(
    image,
    mask,
    color,
    alpha=0.4
):

    overlay = image.copy()

    overlay[mask > 0] = color

    blended = cv2.addWeighted(
        overlay,
        alpha,
        image,
        1 - alpha,
        0
    )

    return blended

In [ ]:
def occlusion_computation(app,img,display_results = False):

    #Face Detection
    img = cv2.resize(img, (512, 512), interpolation=cv2.INTER_CUBIC)
    faces = app.get(img)
    if faces:
        face = faces[0]
        bbox = face.bbox
    else:
        return 0.0
    
    #Gender prediction
    pred_gender = face.sex

    #3D Face estimation (3DDFA-V2)
    boxes = [bbox]
    param_lst, roi_box_lst = tddfa(img, boxes)
    ver_lst = tddfa.recon_vers(
        param_lst,
        roi_box_lst,
        dense_flag=True
    )

    #Surface rendering (face theoritical mask)
    mask_theoretical = np.zeros(
        img.shape[:2],
        dtype=np.uint8
    )
    pts = ver_lst[0][:2, :].T.astype(np.int32)
    hull = cv2.convexHull(pts)
    cv2.fillConvexPoly(
        mask_theoretical,
        hull,
        1
    )

    # Transformation to adjust mask position
    scale_x = 0.9
    scale_y = 1.05
    tx = 15
    ty = -10
    M = np.array([
        [scale_x, 0, tx],
        [0, scale_y, ty]
    ], dtype=np.float32)
    mask_theoretical = cv2.warpAffine(mask_theoretical,M,dsize=img.shape[:2])
    total_pixels = np.sum(mask_theoretical)

    ### BiSeNet
    # Semantic segmentation
    input_tensor = to_tensor(img)
    input_tensor = input_tensor.unsqueeze(0).to(device)
    out = net(input_tensor)[0]
    parsing_b = out.squeeze(0).cpu().numpy().argmax(0)

    # Filtering non-occluded classes
    visible_skin_mask_b = np.isin(
        parsing_b,
        VISIBLE_FACE_CLASSES_B
    ).astype(np.uint8)

    # Filtering occluded classes hair and hat
    hat_b = np.isin(
        parsing_b,
        HAT_B
    ).astype(np.uint8)  
    hair_b = np.isin(
        parsing_b,
        HAIR_B
    ).astype(np.uint8)  

    # Ratio computation
    visible_pixels_b = np.sum(visible_skin_mask_b & mask_theoretical)
    hair_b_in_mask = np.sum(hair_b & mask_theoretical)
    hat_b_in_mask = np.sum(hat_b & mask_theoretical)
    other_b_in_mask = max(0.0,float(total_pixels) - (float(visible_pixels_b)+float(hair_b_in_mask)+float(hat_b_in_mask)))    
    hair_ratio_b = hair_b_in_mask / total_pixels
    hat_ratio_b = hat_b_in_mask / total_pixels
    other_ratio_b = other_b_in_mask / total_pixels

    ### SegFormer
    # Semantic segmentation
    inputs = seg_processor(images=img, return_tensors="pt").to(device)
    outputs = seg_model(**inputs)
    logits = outputs.logits 
    h, w = img.shape[:2]
    image_size = (h, w)
    upsampled_logits = nn.functional.interpolate(logits,
                    size=image_size, # H x W
                    mode='bilinear',
                    align_corners=False)
    labels = upsampled_logits.argmax(dim=1)[0]
    parsing_s = labels.cpu().numpy()

    # Filtering non-occluded classes
    visible_skin_mask_s = np.isin(
        parsing_s,
        VISIBLE_FACE_CLASSES_S
    ).astype(np.uint8)

    # Filtering occluded classes hair and hat
    hat_s = np.isin(
        parsing_s,
        HAT_S
    ).astype(np.uint8)  
    hair_s = np.isin(
        parsing_s,
        HAIR_S
    ).astype(np.uint8)  

    # Ratio computation
    visible_pixels_s = np.sum(visible_skin_mask_s & mask_theoretical)
    hair_s_in_mask = np.sum(hair_s & mask_theoretical)
    hat_s_in_mask = np.sum(hat_s & mask_theoretical)  
    other_s_in_mask = max(0.0,float(total_pixels) - (float(visible_pixels_s)+float(hair_s_in_mask)+float(hat_s_in_mask)))
    hair_ratio_s = hair_s_in_mask / total_pixels
    hat_ratio_s = hat_s_in_mask / total_pixels
    other_ratio_s = other_s_in_mask / total_pixels

    ### Occlusion score
    if pred_gender == 'M':
        occlusion_score = np.clip(M_WEIGHTS["hair_bi"] * hair_ratio_b
        + M_WEIGHTS["hat_bi"] * hat_ratio_b
        + M_WEIGHTS["other_bi"] * other_ratio_b
        + M_WEIGHTS["hair_sf"] * hair_ratio_s
        + M_WEIGHTS["hat_sf"] * hat_ratio_s
        + M_WEIGHTS["other_bg_sf"] * other_ratio_s,
        0,1)
    elif pred_gender == 'F':
        occlusion_score = np.clip(F_WEIGHTS["hair_bi"] * hair_ratio_b
        + F_WEIGHTS["hat_bi"] * hat_ratio_b
        + F_WEIGHTS["other_bi"] * other_ratio_b
        + F_WEIGHTS["hair_sf"] * hair_ratio_s
        + F_WEIGHTS["hat_sf"] * hat_ratio_s
        + F_WEIGHTS["other_bg_sf"] * other_ratio_s,
        0,1)

    # Graphical results
    if display_results:
        # Overlay theoretical face
        vis1 = overlay_mask(
            img,
            mask_theoretical,
            color=(0,255,0),
            alpha=0.35
        )
        # Overlay visible skin 
        vis2 = overlay_mask(
            vis1,
            visible_skin_mask_b,
            color=(255,0,0),
            alpha=0.35
        )
        # Overlay visible skin 
        vis3 = overlay_mask(
            vis2,
            visible_skin_mask_s,
            color=(255,0,0),
            alpha=0.35
        )
        # Plot
        plt.figure(figsize=(8,8))
        plt.imshow(
            cv2.cvtColor(
                vis3,
                cv2.COLOR_BGR2RGB
            )
        )
        plt.axis("off")
        plt.title(
            f"{filename}\n"
            f"GT occlusion={occlusion:.3f} | {gender} - {pred_gender} | Pred occlusion={occlusion_score:.3f}"
        )
        plt.show()

    return occlusion_score

# Generate test predictions

### Make predictions on test dataset

In [ ]:
test_set = Dataset(df_test2, image_dir, training=False)

params_val = {'batch_size': 1,
          'shuffle': False,
          'num_workers': 0}

test_generator = torch.utils.data.DataLoader(test_set, **params_val)

In [ ]:
results_list = []
with torch.inference_mode():
    for batch_idx, (X, y, filename) in tqdm(enumerate(test_generator), total=len(test_generator)):
        for i in range(len(X)):
            img = cv2.imread(os.path.join(image_dir, filename[i]))
            y_pred = occlusion_computation(app,img)
            results_list.append({'filename': filename[i],
                                 'FaceOcclusion': float(y_pred),
                                 'row_id': int(y[i]),
                                 })
results_df = pd.DataFrame(results_list)

In [ ]:
results_df.head()

### Export predictions
Note: We need to add a dummy 'gender' column for the hfactory upload.

In [ ]:
results_df['gender'] = 'x'
results_df.to_csv("test2_predictions.csv", sep=',', index=False)

In [ ]:
df1 = pd.read_csv("test1_predictions.csv")
df2 = pd.read_csv("test2_predictions.csv")

df_final = pd.concat([df1, df2], ignore_index=True)
df_final = df_final.sort_values("row_id")
df_final = df_final[['filename','FaceOcclusion','gender']]
df_final.to_csv("results_complete.csv", index=False)

print(f"Fusion terminée : {len(df_final)} lignes")